## Causal Language Modeling

In [1]:
# causal language models are frequently used for text generation
# predicting next word in the sentence given all the previous words
# evaluate text completed by model on metrics such as: Cross Entropy Loss, Perplexity (Exponential of the Cross Entropy Loss)

![clm_process](clm_process.png)

In [2]:
""" 
Causal language modeling predicts the next token in a sequence of tokens, and the model can only attend to tokens on the left. 
This means the model cannot see future tokens. GPT-2 is an example of a causal language model.
"""

' \nCausal language modeling predicts the next token in a sequence of tokens, and the model can only attend to tokens on the left. \nThis means the model cannot see future tokens. GPT-2 is an example of a causal language model.\n'

In [3]:
""" 
1. Fine-tune DistilGPT2 [https://huggingface.co/distilbert/distilgpt2] on the r/askscience ["https://www.reddit.com/r/askscience/"] 
subset of the ELI5 dataset [https://huggingface.co/datasets/dany0407/eli5_category]
2. Use your finetuned model for inference.
"""

' \n1. Fine-tune DistilGPT2 [https://huggingface.co/distilbert/distilgpt2] on the r/askscience ["https://www.reddit.com/r/askscience/"] \nsubset of the ELI5 dataset [https://huggingface.co/datasets/dany0407/eli5_category]\n2. Use your finetuned model for inference.\n'

In [6]:
!pip install transformers datasets evaluate ipywidgets


  Using cached ipywidgets-8.1.8-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.15-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.8-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.16-py3-none-any.whl (914 kB)
Using cached widgetsnbextension-4.0.15-py3-none-any.whl (2.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [ipywidgets]


In [7]:
from huggingface_hub import notebook_login

notebook_login()

## Load ELI5 dataset

In [8]:
from datasets import load_dataset

eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")


Generating test split: 100%|██████████| 5411/5411 [00:00<00:00, 506514.14 examples/s]


In [9]:
eli5 = eli5.train_test_split(test_size=0.2)

In [12]:
eli5["train"][24]

{'q_id': '7h2yry',
 'title': 'Why over the counter pain medication helps with minor pain but not major pain',
 'selftext': '',
 'category': 'Biology',
 'subreddit': 'explainlikeimfive',
 'answers': {'a_id': ['dqnohzg', 'dqnukx0'],
  'text': ['Over the counter pain medication works by raising the threshold for a stimulus to be felt as pain. Stimuli that are more intense can surpass this higher threshold, being felt as pain anyway. These kinds of medication also have a ceiling effect, meaning higher doses will not yield stronger effects after a certain point. Opioids, on the other hand, work by directly triggering receptors in the patients brain that diminish the sensation of pain, and do not present said ceiling effect, meaning one can always use a higher dosage to get stronger effects (until you OD and die, that is)',
   "It does help with major pain. Tylenol is extremely effective in managing post-operative pain. The fact that it's OTC gives people the perception that it is not very e

## Preprocess

In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")

# extract subfield of text from within the nested structure of answers

eli5 = eli5.flatten()
eli5["train"][24]

/Users/koushalsmodi/Desktop/MachineLearning/MachineLearningProjects/NLP/.venv/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


{'q_id': '7h2yry',
 'title': 'Why over the counter pain medication helps with minor pain but not major pain',
 'selftext': '',
 'category': 'Biology',
 'subreddit': 'explainlikeimfive',
 'answers.a_id': ['dqnohzg', 'dqnukx0'],
 'answers.text': ['Over the counter pain medication works by raising the threshold for a stimulus to be felt as pain. Stimuli that are more intense can surpass this higher threshold, being felt as pain anyway. These kinds of medication also have a ceiling effect, meaning higher doses will not yield stronger effects after a certain point. Opioids, on the other hand, work by directly triggering receptors in the patients brain that diminish the sensation of pain, and do not present said ceiling effect, meaning one can always use a higher dosage to get stronger effects (until you OD and die, that is)',
  "It does help with major pain. Tylenol is extremely effective in managing post-operative pain. The fact that it's OTC gives people the perception that it is not very

In [15]:
def preprocess_function(examples):
    return tokenizer([" ".join(x) for x in examples["answers.text"]])

In [16]:
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched=True,
    num_proc=4,
    remove_columns=eli5["train"].column_names,
)

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (2066 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (2153 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (6294 > 1024). Running this sequence through the model will result in indexing errors
Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]Token indices sequence length is longer than the specified maximum sequence length for this model (1112 > 1024). Running this sequence through the model will result in indexing errors
Token indices sequence length is longer than the specified maximum sequence length for this model (3376 > 1024). Running this sequence

In [17]:
""" 
You can now use a second preprocessing function to:

1. concatenate all the sequences
2. split the concatenated sequences into shorter chunks defined by block_size, which should be both shorter than the maximum input length and short enough for your GPU RAM.
"""

' \nYou can now use a second preprocessing function to:\n\n1. concatenate all the sequences\n2. split the concatenated sequences into shorter chunks defined by block_size, which should be both shorter than the maximum input length and short enough for your GPU RAM.\n'

In [18]:
block_size = 128


def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
    # Split by chunks of block_size.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

In [19]:
lm_dataset = tokenized_eli5.map(group_texts, batched=True, num_proc=4)

Map (num_proc=4): 100%|██████████| 1000/1000 [00:00<00:00, 5774.28 examples/s]


In [20]:
# It’s more efficient to dynamically pad the sentences to the longest length in a batch during collation, instead of padding the whole dataset to the maximum length.

from transformers import DataCollatorForLanguageModeling

tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

## Train

In [21]:
from transformers import AutoModelForCausalLM, TrainingArguments, Trainer

model = AutoModelForCausalLM.from_pretrained("distilbert/distilgpt2")

W0404 19:53:08.852000 84261 torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [25]:
training_args = TrainingArguments(
    output_dir="my_awesome_eli5_clm-model",
    eval_strategy="epoch",
    learning_rate=2e-5,
    weight_decay=0.01,
    push_to_hub=True,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_dataset["train"],
    eval_dataset=lm_dataset["test"],
    data_collator=data_collator,
    tokenizer=tokenizer
)

trainer.train()

 13%|█▎        | 500/3999 [03:29<26:28,  2.20it/s]

{'loss': 3.9691, 'grad_norm': 4.158233642578125, 'learning_rate': 1.749937484371093e-05, 'epoch': 0.38}


 25%|██▌       | 1000/3999 [07:56<27:35,  1.81it/s]

{'loss': 3.9551, 'grad_norm': 3.342378854751587, 'learning_rate': 1.4998749687421857e-05, 'epoch': 0.75}


 33%|███▎      | 1333/3999 [11:19<39:38,  1.12it/s]/Users/koushalsmodi/Desktop/MachineLearning/MachineLearningProjects/NLP/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
                                                   
 33%|███▎      | 1333/3999 [12:09<39:38,  1.12it/s]

...

 88%|████████▊ | 3500/3999 [31:50<04:18,  1.93it/s]

{'loss': 3.8187, 'grad_norm': 3.8715779781341553, 'learning_rate': 2.4956239059764944e-06, 'epoch': 2.63}


100%|██████████| 3999/3999 [36:19<00:00,  2.42it/s]/Users/koushalsmodi/Desktop/MachineLearning/MachineLearningProjects/NLP/.venv/lib/python3.11/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
                                                   
100%|██████████| 3999/3999 [37:05<00:00,  1.80it/s]

{'eval_loss': 3.815232992172241, 'eval_runtime': 45.4792, 'eval_samples_per_second': 53.299, 'eval_steps_per_second': 6.662, 'epoch': 3.0}
{'train_runtime': 2225.0789, 'train_samples_per_second': 14.368, 'train_steps_per_second': 1.797, 'train_loss': 3.8745727691688545, 'epoch': 3.0}


TrainOutput(global_step=3999, training_loss=3.8745727691688545, metrics={'train_runtime': 2225.0789, 'train_samples_per_second': 14.368, 'train_steps_per_second': 1.797, 'total_flos': 1044239801647104.0, 'train_loss': 3.8745727691688545, 'epoch': 3.0})

In [26]:
import math

eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

100%|██████████| 303/303 [00:44<00:00,  6.84it/s]

Perplexity: 45.39


## Inference

In [27]:
prompt = "Somatic hypermutation allows the immune system to"

In [28]:
from transformers import pipeline

generator = pipeline("text-generation", model="koushalsmodi/my_awesome_eli5_clm-model")
generator(prompt)

[{'generated_text': 'Somatic hypermutation allows the immune system to adapt to a more drastic response. When the immune system is overwhelmed, the immune system does NOT adapt to the problem and eventually starts firing. This is why there is a natural imbalance in the immune system'}]

In [29]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("koushalsmodi/my_awesome_eli5_clm-model")
inputs = tokenizer(prompt, return_tensors="pt").input_ids

In [30]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained("koushalsmodi/my_awesome_eli5_clm-model")
outputs = model.generate(inputs, max_new_tokens=100, do_sample=True, top_k=50, top_p=0.95)

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [31]:
tokenizer.batch_decode(outputs, skip_special_tokens=True)

["Somatic hypermutation allows the immune system to react more effectively to the environment, and to detect disease or illness by recognizing it as a sign of its own, not the other way around. We are very well aware that the immune system has a lot of specific functions to help keep us alive. There are several different immune systems, so you can make the decision, to make a choice based on the environment, which might work, or the one you think would be best to use to limit inflammation. When you've just had these two choices,"]